In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import zipfile
import pandas as pd

DRIVE_FOLDER_PATH = "/content/drive/MyDrive/NLP/Assignment 3/"

# Unzipping dataset.zip
dataset_zip_path = os.path.join(DRIVE_FOLDER_PATH, "dataset.zip")
dataset_extract_path = os.path.join(DRIVE_FOLDER_PATH, "dataset")

if not os.path.exists(dataset_extract_path):
    with zipfile.ZipFile(dataset_zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_extract_path)
    print(f"Extracted zip file to {dataset_extract_path}")
else:
    print(f"Zip file already extracted at {dataset_extract_path}")

# Loading the data files and normalizing column names to lowercase
def load_and_normalize(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.lower()
    return df

train_path = os.path.join(dataset_extract_path, "train.csv")
val_path = os.path.join(dataset_extract_path, "val.csv")
test_path = os.path.join(dataset_extract_path, "test.csv")

train_df = load_and_normalize(train_path)
val_df = load_and_normalize(val_path)
test_df = load_and_normalize(test_path)

print(f"Train shape: {train_df.shape}")
print(f"Val shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

# Preprocess labels
def map_label(rating):
    if rating in [4, 5]:
        return 2  # positive
    elif rating == 3:
        return 1  # neutral
    else:
        return 0  # negative

for df in [train_df, val_df, test_df]:
    df['label'] = df['label'].apply(map_label)

print("Train label distribution:\n", train_df['label'].value_counts())
print("Val label distribution:\n", val_df['label'].value_counts())
print("Test label distribution:\n", test_df['label'].value_counts())

# Saving the preprocessed dataframes
train_df.to_csv(os.path.join(dataset_extract_path, "train_preprocessed.csv"), index=False)
val_df.to_csv(os.path.join(dataset_extract_path, "val_preprocessed.csv"), index=False)
test_df.to_csv(os.path.join(dataset_extract_path, "test_preprocessed.csv"), index=False)


Zip file already extracted at /content/drive/MyDrive/NLP/Assignment 3/dataset
Train shape: (10000, 2)
Val shape: (2000, 2)
Test shape: (2000, 2)
Train label distribution:
 label
0    5955
2    2054
1    1991
Name: count, dtype: int64
Val label distribution:
 label
0    1150
2     445
1     405
Name: count, dtype: int64
Test label distribution:
 label
0    1231
2     390
1     379
Name: count, dtype: int64


In [3]:
# Part 2, Soft Prompt Tuning Implementation

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel
import pandas as pd
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

DRIVE_FOLDER_PATH = "/content/drive/MyDrive/NLP/Assignment 3/"
dataset_extract_path = DRIVE_FOLDER_PATH + "dataset/"

# Parameters
PRETRAINED_MODEL = 'bert-base-uncased'
BATCH_SIZE = 32
MAX_LEN = 128
NUM_CLASSES = 3
SOFT_PROMPT_TOKENS = 10
EPOCHS = 10  # Stopping early if validation stops

# Loading Preprocessed Data
train_df = pd.read_csv(dataset_extract_path + "train_preprocessed.csv")
val_df = pd.read_csv(dataset_extract_path + "val_preprocessed.csv")
test_df = pd.read_csv(dataset_extract_path + "test_preprocessed.csv")

# Dataset Class
class ReviewDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
            add_special_tokens=True
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Tokenizer & Datasets
tokenizer = BertTokenizer.from_pretrained(PRETRAINED_MODEL)
train_ds = ReviewDataset(train_df, tokenizer, MAX_LEN)
val_ds = ReviewDataset(val_df, tokenizer, MAX_LEN)
test_ds = ReviewDataset(test_df, tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Our Model
class SoftPromptBERT(nn.Module):
    def __init__(self, bert_model_name, num_classes, soft_prompt_tokens=10):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        for param in self.bert.parameters():
            param.requires_grad = False
        self.hidden_size = self.bert.config.hidden_size
        self.soft_prompts = nn.Parameter(torch.randn(soft_prompt_tokens, self.hidden_size))
        self.classifier = nn.Linear(self.hidden_size, num_classes)
    def forward(self, input_ids, attention_mask, token_type_ids):
        batch_size = input_ids.size(0)
        device = input_ids.device
        # Get BERT input embeddings
        inputs_embeds = self.bert.embeddings(input_ids=input_ids)
        # Expand soft prompts for batch
        soft_prompt = self.soft_prompts.unsqueeze(0).expand(batch_size, -1, -1)
        # Concatenate [soft prompts] + [BERT embeddings]
        concat_embeds = torch.cat([soft_prompt, inputs_embeds], dim=1)
        # Adjust attention mask: prepend 1s for soft prompts
        soft_attention = torch.ones((batch_size, self.soft_prompts.shape[0]), dtype=attention_mask.dtype, device=device)
        concat_attention_mask = torch.cat([soft_attention, attention_mask], dim=1)
        # Pass through BERT (using embeddings directly)
        outputs = self.bert(
            inputs_embeds=concat_embeds,
            attention_mask=concat_attention_mask,
            token_type_ids=None  # token_type_ids are not used for classification here
        )
        cls_output = outputs.last_hidden_state[:, self.soft_prompts.shape[0], :]
        logits = self.classifier(cls_output)
        return logits

model = SoftPromptBERT(PRETRAINED_MODEL, NUM_CLASSES, SOFT_PROMPT_TOKENS).to(DEVICE)

# Only train soft prompts and classifier that asked
optimizer = torch.optim.AdamW([
    {'params': model.soft_prompts},
    {'params': model.classifier.parameters()}
], lr=2e-4, betas=(0.9, 0.999), weight_decay=1e-2)

criterion = nn.CrossEntropyLoss()
# (For soft prompts, in most works only the classifier loss is used; for compliance, just reuse classifier loss)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        token_type_ids = batch.get('token_type_ids', None)
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask, token_type_ids)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            token_type_ids = batch.get('token_type_ids', None)
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            logits = model(input_ids, attention_mask, token_type_ids)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), np.array(all_preds), np.array(all_labels)

# Training Loop
best_val_loss = float('inf')
best_model_state = None
for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict()
        torch.save(best_model_state, dataset_extract_path + 'best_soft_prompt_model.pt')
    # Early stopping (patience=2)
    if epoch > 1 and val_loss > best_val_loss:
        print("Early stopping triggered.")
        break

# Test Evaluation
model.load_state_dict(torch.load(dataset_extract_path + 'best_soft_prompt_model.pt'))
_, test_preds, test_labels = evaluate(model, test_loader, criterion)

def accuracy(preds, labels):
    return (preds == labels).mean()

def precision_recall_f1(preds, labels, num_classes=3):
    prec = []
    rec = []
    f1 = []
    for cls in range(num_classes):
        tp = np.sum((preds == cls) & (labels == cls))
        fp = np.sum((preds == cls) & (labels != cls))
        fn = np.sum((preds != cls) & (labels == cls))
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        prec.append(p)
        rec.append(r)
        f1.append(f)
    return prec, rec, f1

test_acc = accuracy(test_preds, test_labels)
test_prec, test_rec, test_f1 = precision_recall_f1(test_preds, test_labels, num_classes=NUM_CLASSES)

print(f"Test Accuracy: {test_acc:.4f}")
for i, label in enumerate(["Negative", "Neutral", "Positive"]):
    print(f"{label}: Precision={test_prec[i]:.4f} Recall={test_rec[i]:.4f} F1={test_f1[i]:.4f}")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Epoch 1: Train Loss = 0.9071 | Val Loss = 0.8425
Epoch 2: Train Loss = 0.8114 | Val Loss = 0.7687
Epoch 3: Train Loss = 0.7670 | Val Loss = 0.7344
Epoch 4: Train Loss = 0.7373 | Val Loss = 0.7124
Epoch 5: Train Loss = 0.7154 | Val Loss = 0.6943
Epoch 6: Train Loss = 0.7046 | Val Loss = 0.6818
Epoch 7: Train Loss = 0.6926 | Val Loss = 0.6683
Epoch 8: Train Loss = 0.6801 | Val Loss = 0.6607
Epoch 9: Train Loss = 0.6777 | Val Loss = 0.6574
Epoch 10: Train Loss = 0.6714 | Val Loss = 0.6516
Test Accuracy: 0.7190
Negative: Precision=0.7878 Recall=0.9017 F1=0.8409
Neutral: Precision=0.4850 Recall=0.2559 F1=0.3351
Positive: Precision=0.5908 Recall=0.5923 F1=0.5915
